# Customer-facing support agent — tools on a Snowflake MCP server

Four tools on real Snowflake objects, exposed through one MCP server:

| Tool | Snowflake object | Type |
|---|---|---|
| `classify_intent` | `SUPPORT.CLASSIFY_INTENT` → fine-tuned Llama 3.1 8B | SQL UDF |
| `search_reviews` | `SUPPORT.SEARCH_REVIEWS` | Cortex Search |
| `get_order_status` | `SUPPORT.GET_ORDER_STATUS` | SQL UDF |
| `file_ticket` | `SUPPORT.FILE_TICKET` (routes by intent) → `SUPPORT.TICKETS` | Stored procedure |

## 1. The data

In [ ]:
-- Public product reviews (what search_reviews searches)
SELECT PRODUCT, RATING, REVIEW_TEXT, REVIEW_DATE
FROM MCP_HOL.SUPPORT.REVIEWS
WHERE PRODUCT = 'Summit Winter Jacket'
ORDER BY REVIEW_DATE DESC
LIMIT 10

In [ ]:
-- Orders (what get_order_status looks up)
SELECT * FROM MCP_HOL.SUPPORT.ORDERS ORDER BY ORDER_ID

In [ ]:
-- Support incidents log (what file_ticket writes to)
SELECT * FROM MCP_HOL.SUPPORT.TICKETS ORDER BY CREATED_AT DESC

In [ ]:
-- Routing policy: intent label -> queue / priority / SLA.
-- This is why classify_intent matters: file_ticket routes on the label, and the
-- model can't invent your queues or SLAs.
SELECT * FROM MCP_HOL.SUPPORT.INTENT_ROUTING

## 2. Tool 1 — `classify_intent`

A SQL UDF that calls an **in-account fine-tuned Llama 3.1 8B model** (`SUPPORT_INTENT_8B`) to triage a customer's message into one intent label. That label is what `file_ticket` uses to route the incident — so this call is load-bearing, not decorative.

**Agent input** (`input_schema` exposed by the MCP server):
```json
{ "message": "string — the customer's message" }
```
**Agent sees back:** one intent label string — e.g. `DEFECTIVE_ITEM`.

In [ ]:
CREATE OR REPLACE FUNCTION MCP_HOL.SUPPORT.CLASSIFY_INTENT(MESSAGE VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
  UPPER(TRIM(SNOWFLAKE.CORTEX.COMPLETE(
    'MCP_HOL.SUPPORT.SUPPORT_INTENT_8B',
    'Classify the customer support message into exactly one label from this list: '
    || 'ORDER_STATUS, SHIPPING_DELAY, DEFECTIVE_ITEM, RETURN_REFUND, SIZING_EXCHANGE, GENERAL_FEEDBACK. '
    || 'Reply with only the label and nothing else. Message: ' || message || ' Label:'
  )))
$$

In [ ]:
-- input: a customer message   ->   output: one intent label
SELECT MCP_HOL.SUPPORT.CLASSIFY_INTENT('The zipper broke the first time I wore the jacket') AS output

## 3. Tool 2 — `search_reviews`

A Cortex Search service over `REVIEWS`. The agent sends a natural-language query; it returns the most relevant reviews.

**Agent input** (auto-generated by the MCP server for a Cortex Search tool):
```json
{ "query": "string — what to search for, e.g. 'zipper keeps breaking'" }
```
**Agent sees back:** the top matching reviews as JSON — `REVIEW_TEXT` plus attributes (`PRODUCT`, `RATING`, `ORDER_ID`, `REVIEW_DATE`) and relevance `@scores`, e.g.
```json
{ "results": [ { "REVIEW_TEXT": "Broken zipper after two uses...", "PRODUCT": "Summit Winter Jacket", "RATING": "1" } ] }
```

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE MCP_HOL.SUPPORT.SEARCH_REVIEWS
  ON REVIEW_TEXT
  ATTRIBUTES PRODUCT, RATING, ORDER_ID, REVIEW_DATE
  WAREHOUSE = COCO_WH
  TARGET_LAG = '1 hour'
  AS SELECT REVIEW_TEXT, PRODUCT, RATING, ORDER_ID, REVIEW_DATE
     FROM MCP_HOL.SUPPORT.REVIEWS

In [ ]:
-- What the tool runs under the hood.  input: a query string
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'MCP_HOL.SUPPORT.SEARCH_REVIEWS',
  '{"query":"zipper keeps breaking","columns":["REVIEW_TEXT","PRODUCT","RATING"],"limit":5}'
) AS output

## 4. Tool 3 — `get_order_status`

A SQL UDF that returns one order's status. Owner's rights — the caller runs it without reading the `ORDERS` table.

**Agent input** (`input_schema` exposed by the MCP server):
```json
{ "order_id": "string — e.g. ORD-10001" }
```
**Agent sees back:** one line of text — e.g. `Order ORD-10001 (Summit Winter Jacket): Delivered on 2026-01-05 via UPS, tracking 1Z999AA10123456784.`

In [ ]:
CREATE OR REPLACE FUNCTION MCP_HOL.SUPPORT.GET_ORDER_STATUS(ORDER_ID VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
WITH input AS (SELECT ORDER_ID AS oid)
SELECT COALESCE(
  (SELECT 'Order ' || o.ORDER_ID || ' (' || o.PRODUCT || '): ' || o.STATUS ||
          CASE WHEN o.STATUS = 'Delivered'  THEN ' on ' || TO_VARCHAR(o.EST_DELIVERY, 'YYYY-MM-DD')
               WHEN o.STATUS = 'In Transit' THEN ', expected ' || TO_VARCHAR(o.EST_DELIVERY, 'YYYY-MM-DD')
               ELSE '' END ||
          CASE WHEN o.CARRIER IS NOT NULL THEN ' via ' || o.CARRIER || ', tracking ' || o.TRACKING_NO ELSE '' END ||
          '.'
   FROM MCP_HOL.SUPPORT.ORDERS o, input i
   WHERE o.ORDER_ID = i.oid),
  'No order found for ' || (SELECT oid FROM input) || '.')
$$

In [ ]:
-- input: an order id   ->   output: one line for that order
SELECT MCP_HOL.SUPPORT.GET_ORDER_STATUS('ORD-10001') AS output

## 5. Tool 4 — `file_ticket`

A stored procedure that files a ServiceNow-style incident and logs it to `TICKETS`. It takes the **`intent` label from `classify_intent`** and looks up `INTENT_ROUTING` to set the queue, priority, and SLA — so the fine-tuned model's output decides where the case goes. Self-contained, owner's rights.

**Agent input** (`input_schema` exposed by the MCP server):
```json
{ "order_id": "string — e.g. ORD-10001",
  "issue": "string — short description of the problem",
  "intent": "string — the label from classify_intent, e.g. DEFECTIVE_ITEM" }
```
**Agent sees back:** a confirmation with the incident number and routing — e.g. `Created ServiceNow incident INC0001001 for order ORD-10001 — classified as DEFECTIVE_ITEM, routed to the Product Quality queue at priority P2 (SLA 8h).`

In [ ]:
CREATE OR REPLACE PROCEDURE MCP_HOL.SUPPORT.FILE_TICKET(ORDER_ID STRING, ISSUE STRING, INTENT STRING)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = 3.11
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
EXECUTE AS OWNER
AS $$
def run(session, order_id, issue, intent):
    label = (intent or '').strip().upper()
    row = session.sql(
        'SELECT QUEUE, PRIORITY, SLA_HOURS FROM MCP_HOL.SUPPORT.INTENT_ROUTING WHERE INTENT = ?',
        params=[label]).collect()
    if row:
        queue, priority, sla = row[0][0], row[0][1], row[0][2]
    else:
        label, queue, priority, sla = 'UNCLASSIFIED', 'General Triage', 'P3', 24
    seq = session.sql('SELECT MCP_HOL.SUPPORT.TICKET_SEQ.NEXTVAL').collect()[0][0]
    inc = 'INC' + str(seq).zfill(7)
    session.sql(
        'INSERT INTO MCP_HOL.SUPPORT.TICKETS '
        '(INCIDENT_NUMBER, ORDER_ID, ISSUE, INTENT, QUEUE, PRIORITY, STATUS, CREATED_AT) '
        "VALUES (?, ?, ?, ?, ?, ?, 'New', CURRENT_TIMESTAMP())",
        params=[inc, order_id, issue, label, queue, priority]).collect()
    return ('Created ServiceNow incident ' + inc + ' for order ' + str(order_id)
            + ' — classified as ' + label + ', routed to the ' + queue
            + ' queue at priority ' + priority + ' (SLA ' + str(sla) + 'h).')
$$

In [ ]:
-- input: order id + issue + intent (from classify_intent)   ->   routed incident
CALL MCP_HOL.SUPPORT.FILE_TICKET('ORD-10001', 'Zipper broke on first wear', 'DEFECTIVE_ITEM')

## 6. Create the MCP server

One statement exposes the four objects above as agent tools.

In [ ]:
CREATE OR REPLACE MCP SERVER MCP_HOL.AGENTS.MCP_HOL
FROM SPECIFICATION $$
tools:
  - name: "classify_intent"
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.CLASSIFY_INTENT"
    title: "Classify customer intent"
    description: "Classify a customer's message into exactly one support intent label (ORDER_STATUS, SHIPPING_DELAY, DEFECTIVE_ITEM, RETURN_REFUND, SIZING_EXCHANGE, GENERAL_FEEDBACK) using a fine-tuned model. Call this FIRST to triage the customer's message; its label is required to file a ticket."
    config:
      type: "function"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          message: { type: "string", description: "The customer's message to classify" }
        required: ["message"]
  - name: "search_reviews"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "MCP_HOL.SUPPORT.SEARCH_REVIEWS"
    title: "Search customer reviews"
    description: "Search customer product reviews for a topic (zipper problems, sizing, shipping, warmth, etc.) and return the most relevant matching reviews."
  - name: "get_order_status"
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.GET_ORDER_STATUS"
    title: "Look up order status"
    description: "Look up the status and shipment details of a SINGLE order by its order id. Returns the current status, carrier, tracking number, and delivery date."
    config:
      type: "function"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          order_id: { type: "string", description: "The customer's order id, e.g. ORD-10001" }
        required: ["order_id"]
  - name: "file_ticket"
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.FILE_TICKET"
    title: "Open a ServiceNow incident"
    description: "File a ServiceNow incident for the customer's order. Requires the intent label from classify_intent, which routes the incident to the right queue/priority. Returns the incident number and routing."
    config:
      type: "procedure"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          order_id: { type: "string", description: "The affected order id, e.g. ORD-10001" }
          issue: { type: "string", description: "Short description of the customer's problem" }
          intent: { type: "string", description: "The intent label from classify_intent (e.g. DEFECTIVE_ITEM). Used to route the incident." }
        required: ["order_id", "issue", "intent"]
$$;